# NBA Statistics and Salaries

Salaries from https://www.kaggle.com/datasets/omarsobhy14/nba-players-salaries

Stats from www.basketball-reference.com obtained with basketball_reference_web_scraper

From them we get every player for which we have the data:
- name,
- positions,	
- age,	
- team,	
- season,	
- ppg,	
- fg_avg,	
- three_pt_avg,	
- ft_avg,	
- apg,	
- rpg,	
- spg,	
- bpg,	
- mpg,	
- salary.



In [1]:
import pandas as pd
import unicodedata

## Name normalization function
Some names can be written in more than one way due to special charcters. Most popular example would be Nikola Jokić whose name is ofter written "Jokic" by english speaking media. This normalization takes care of that fact.

In [2]:
def normalize_name(s):
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.strip()

## Salaries per season

In [3]:
salaries = pd.read_csv('../data/raw/salaries.csv')
salaries.head()

,Player Id,Player Name,2022/2023,2023/2024,2024/2025,2025/2026
0,1,Stephen Curry,"$48,070,014","$51,915,615","$55,761,217","$59,606,817"
1,2,John Wall,"$47,345,760",$0,$0,$0
2,3,Russell Westbrook,"$47,080,179",$0,$0,$0
3,4,LeBron James,"$44,474,988","$46,698,737","$50,434,636",$0
4,5,Kevin Durant,"$44,119,845","$47,649,433","$51,179,020","$54,708,608"


## Cleaning and transformation of salaries

In [4]:
season_salaries = salaries.melt(
    id_vars=["Player Name"],              # columns to keep
    value_vars=["2022/2023","2023/2024","2024/2025","2025/2026"],  # season columns
    var_name="Season",
    value_name="Salary"
)

season_salaries["Salary"] = season_salaries["Salary"].replace('[\$,]', '', regex=True).astype(float)
season_salaries = season_salaries[season_salaries["Salary"] != 0]
season_salaries["Start Year"] = season_salaries["Season"].str.slice(0,4).astype(str)
season_salaries = season_salaries.drop(columns=["Season"])
season_salaries = season_salaries.rename(columns={"Start Year": "season", "Player Name": "name", "Salary": "salary"}) 
season_salaries["name"] = season_salaries["name"].apply(normalize_name)

# we'll remove salaries that are under the minimum salary for each year.
# Players excluded are those that earned less than the rookies.
# info from https://www.spotrac.com/nba/cba/minimum
min_salary = {
    "2022": 1017781,
    "2023": 1119563,	
    "2024": 1157153,	
    "2025": 1272870
}
season_salaries["min_salary"] = season_salaries["season"].map(min_salary)
season_salaries = season_salaries[season_salaries["salary"] > season_salaries["min_salary"]]
season_salaries = season_salaries.drop(columns="min_salary")

In [5]:
season_salaries.sample(10)

,name,salary,season
1741,Zach LaVine,45999660.0,2025
120,Robert Covington,12307692.0,2022
1097,Jared Rhoden,1761752.0,2023
1937,Tyler Herro,31000000.0,2025
1114,Mac McClung,1761752.0,2023
923,Andrew Nembhard,2131905.0,2023
417,Xavier Tillman,1782621.0,2022
17,Kemba Walker,37281261.0,2022
628,John Collins,25340000.0,2023
294,Jake LaRavia,3047640.0,2022


In [6]:
season_salaries.shape

(1209, 3)

## Obtaining the statistics

In [ ]:
# pip install basketball_reference_web_scraper
from basketball_reference_web_scraper import client
from basketball_reference_web_scraper.data import OutputType

end_years = [ 2022, 2023, 2024, 2025 ]
for year in end_years:
    client.players_season_totals(
        season_end_year=year, 
        output_type=OutputType.CSV, 
        output_file_path=f"../data/raw/{year-1}_{year}_player_season_totals.csv"
    )

In [7]:
stats2022 = pd.read_csv('../data/raw/2021_2022_player_season_totals.csv')
stats2023 = pd.read_csv('../data/raw/2022_2023_player_season_totals.csv')
stats2024 = pd.read_csv('../data/raw/2023_2024_player_season_totals.csv')
stats2025 = pd.read_csv('../data/raw/2024_2025_player_season_totals.csv')

stats2022['season'] = '2022'
stats2023['season'] = '2023'
stats2024['season'] = '2024'
stats2025['season'] = '2025'

full_stats = pd.concat([stats2022, stats2023, stats2024, stats2025], ignore_index=True) 
full_stats.head()

,slug,name,positions,age,team,games_played,games_started,minutes_played,made_field_goals,attempted_field_goals,...,attempted_free_throws,offensive_rebounds,defensive_rebounds,assists,steals,blocks,turnovers,personal_fouls,points,season
0,youngtr01,Trae Young,POINT GUARD,23,ATLANTA HAWKS,76,76,2652,711,1544,...,553,50,234,737,72,7,303,128,2155,2022
1,derozde01,DeMar DeRozan,POWER FORWARD,32,CHICAGO BULLS,76,76,2743,774,1535,...,593,56,336,374,68,24,181,178,2118,2022
2,embiijo01,Joel Embiid,CENTER,27,PHILADELPHIA 76ERS,68,68,2297,666,1334,...,803,146,650,284,77,99,214,181,2079,2022
3,tatumja01,Jayson Tatum,SMALL FORWARD,23,BOSTON CELTICS,76,76,2731,708,1564,...,469,85,524,334,75,49,217,174,2046,2022
4,jokicni01,Nikola Jokić,CENTER,26,DENVER NUGGETS,74,74,2476,764,1311,...,468,206,813,584,109,63,281,191,2004,2022


In [8]:
full_stats.dtypes

slug                                 object
name                                 object
positions                            object
age                                   int64
team                                 object
games_played                          int64
games_started                         int64
minutes_played                        int64
made_field_goals                      int64
attempted_field_goals                 int64
made_three_point_field_goals          int64
attempted_three_point_field_goals     int64
made_free_throws                      int64
attempted_free_throws                 int64
offensive_rebounds                    int64
defensive_rebounds                    int64
assists                               int64
steals                                int64
blocks                                int64
turnovers                             int64
personal_fouls                        int64
points                                int64
season                          

In [9]:
# some playes played for more than one team the season prior to the salary they recieved
# we'll remove these players, we want to only calculate the next season salary based on a full year performance 
# and their situation tends to be more complex
full_stats = full_stats[
    full_stats.duplicated(['name', 'season'], keep=False) == False
]

full_stats["name"] = full_stats["name"].apply(normalize_name)


full_stats["ppg"] = full_stats["points"] / full_stats["games_played"]
full_stats["fg_avg"] = full_stats["made_field_goals"] / full_stats["attempted_field_goals"]
full_stats["three_pt_avg"] = full_stats["made_three_point_field_goals"] / full_stats["attempted_three_point_field_goals"]
full_stats["ft_avg"] = full_stats["made_free_throws"] / full_stats["attempted_free_throws"]

full_stats["apg"] = full_stats["assists"] / full_stats["games_played"]
full_stats["rpg"] = (full_stats["offensive_rebounds"] + full_stats["defensive_rebounds"]) / full_stats["games_played"]
full_stats["spg"] = full_stats["steals"] / full_stats["games_played"]
full_stats["bpg"] = full_stats["blocks"] / full_stats["games_played"]

full_stats["mpg"] = full_stats["minutes_played"] / full_stats["games_played"]

full_stats = full_stats.drop(columns=[ 
        'games_played', 'games_started', 'minutes_played', 'made_field_goals',
        'attempted_field_goals', 'made_three_point_field_goals',
        'attempted_three_point_field_goals', 'made_free_throws',
        'attempted_free_throws', 'offensive_rebounds', 'defensive_rebounds',
        'assists', 'steals', 'blocks', 'turnovers', 'personal_fouls', 'points' ])

full_stats = full_stats.fillna(0)

In [10]:
full_stats.sample(10)

,slug,name,positions,age,team,season,ppg,fg_avg,three_pt_avg,ft_avg,apg,rpg,spg,bpg,mpg
2396,josepco01,Cory Joseph,POINT GUARD,33,ORLANDO MAGIC,2025,3.540000,0.402597,0.364486,0.777778,1.440000,1.480000,0.520000,0.080000,12.240000
2205,suggsja01,Jalen Suggs,POINT GUARD,23,ORLANDO MAGIC,2025,16.200000,0.409563,0.314050,0.881818,3.685714,4.028571,1.457143,0.942857,28.628571
1449,robindu01,Duncan Robinson,SMALL FORWARD,29,MIAMI HEAT,2024,12.867647,0.450370,0.395349,0.888889,2.823529,2.544118,0.676471,0.235294,28.014706
222,wagnemo01,Moritz Wagner,CENTER,24,ORLANDO MAGIC,2022,8.952381,0.497475,0.328358,0.806202,1.380952,3.682540,0.317460,0.206349,15.238095
2371,adamsst01,Steven Adams,CENTER,31,HOUSTON ROCKETS,2025,3.879310,0.544910,0.000000,0.462366,1.137931,5.637931,0.379310,0.482759,13.689655
1658,smartma01,Marcus Smart,POINT GUARD,29,MEMPHIS GRIZZLIES,2024,14.450000,0.430380,0.313433,0.767857,4.300000,2.650000,2.050000,0.250000,30.250000
2276,diabamo01,Moussa Diabate,CENTER,23,CHARLOTTE HORNETS,2025,5.676056,0.596429,0.000000,0.594828,0.788732,6.169014,0.647887,0.563380,17.478873
1059,dunnkr01,Kris Dunn,POINT GUARD,28,UTAH JAZZ,2023,13.181818,0.537037,0.472222,0.773585,5.636364,4.545455,1.136364,0.454545,25.818182
661,seldewa01,Wayne Selden,SHOOTING GUARD,27,NEW YORK KNICKS,2022,1.666667,0.250000,0.500000,0.500000,0.333333,0.333333,0.000000,0.000000,6.333333
416,fultzma01,Markelle Fultz,POINT GUARD,23,ORLANDO MAGIC,2022,10.833333,0.474286,0.235294,0.806452,5.500000,2.722222,1.111111,0.277778,20.000000


In [11]:
full_stats.shape

(1959, 15)

Another problem - some names are written with different punctuation, suffixes, and abbreviations (Jr, Jr., III, accents, spaces, hyphens, etc.) despite being the same name

In [12]:
import re

def normalize_punctuation(s):
    s = s.lower()
    s = re.sub(r'\.', '', s)      # remove periods
    s = re.sub(r',', '', s)       # remove commas
    s = re.sub(r'\s+', ' ', s)    # collapse spaces
    s = s.strip()
    return s

full_stats['name_norm'] = full_stats['name'].apply(normalize_punctuation)
season_salaries['name_norm'] = season_salaries['name'].apply(normalize_punctuation)

full_data = pd.merge(full_stats, season_salaries, on=["name_norm", "season"], how="inner")

In [157]:
full_data.sample(10)

,id,name_x,positions,age,team,season,ppg,fg_avg,three_pt_avg,ft_avg,apg,rpg,spg,bpg,mpg,name_norm,name_y,salary
269,leesa01,Saben Lee,POINT GUARD,22,DETROIT PISTONS,2022,5.621622,0.389831,0.233333,0.788732,2.891892,2.351351,0.972973,0.324324,16.324324,saben lee,Saben Lee,2266640.0
384,leonaka01,Kawhi Leonard,SMALL FORWARD,31,LOS ANGELES CLIPPERS,2023,23.826923,0.512055,0.416000,0.870968,3.923077,6.500000,1.384615,0.538462,33.615385,kawhi leonard,Kawhi Leonard,45640084.0
115,poweldw01,Dwight Powell,CENTER,30,DALLAS MAVERICKS,2022,8.743902,0.670823,0.351351,0.783019,1.182927,4.926829,0.451220,0.475610,21.926829,dwight powell,Dwight Powell,11080125.0
411,dortlu01,Luguentz Dort,SMALL FORWARD,23,OKLAHOMA CITY THUNDER,2023,13.689189,0.388316,0.330073,0.772201,2.081081,4.648649,1.013514,0.310811,30.702703,luguentz dort,Luguentz Dort,15277778.0
700,portibo01,Bobby Portis,POWER FORWARD,28,MILWAUKEE BUCKS,2024,13.780488,0.507625,0.406504,0.790323,1.268293,7.402439,0.829268,0.414634,24.487805,bobby portis,Bobby Portis,12578286.0
708,simonan01,Anfernee Simons,SHOOTING GUARD,24,PORTLAND TRAIL BLAZERS,2024,22.586957,0.430108,0.385185,0.915730,5.543478,3.630435,0.500000,0.108696,34.391304,anfernee simons,Anfernee Simons,25892857.0
671,jacksja02,Jaren Jackson Jr.,CENTER,24,MEMPHIS GRIZZLIES,2024,22.515152,0.444062,0.319672,0.808153,2.333333,5.530303,1.212121,1.606061,32.181818,jaren jackson jr,Jaren Jackson Jr,25257798.0
534,moodymo01,Moses Moody,SHOOTING GUARD,20,GOLDEN STATE WARRIORS,2023,4.793651,0.475771,0.362963,0.698113,0.809524,1.666667,0.285714,0.111111,12.968254,moses moody,Moses Moody,3918480.0
397,anunoog01,OG Anunoby,SMALL FORWARD,25,TORONTO RAPTORS,2023,16.776119,0.475706,0.386921,0.838323,1.955224,4.955224,1.910448,0.746269,35.611940,og anunoby,OG Anunoby,18642857.0
298,iguodan01,Andre Iguodala,SMALL FORWARD,38,GOLDEN STATE WARRIORS,2022,4.000000,0.380165,0.229730,0.750000,3.677419,3.225806,0.870968,0.709677,19.451613,andre iguodala,Andre Iguodala,2905851.0


Jaren Jackson Jr. is an example of this.

In [13]:
full_data.shape

(927, 18)

In [14]:
full_data["name"] = full_data["name_x"]
full_data = full_data.drop(columns=["name_x","name_y","name_norm"])
full_data.to_csv('../data/raw/nba_full_data.csv', index=False)
full_data.to_parquet('../data/processed/clean_nba.parquet', index=False)

In [15]:
full_data["positions"].value_counts()

positions
SHOOTING GUARD    230
POWER FORWARD     187
CENTER            187
SMALL FORWARD     167
POINT GUARD       156
Name: count, dtype: int64